# 02 · Hierarchy and time-irreversibility

Visualises `src/hierarchy` output (run `python -m src.hierarchy.run`). Model-based FDT deviation (Acero-Pousa et al.
2025) and model-free pairwise time-irreversibility, across HC / AP / NAP, per network, and vs symptoms.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.core import paths
from src.core.data import GROUPS, load_data, load_model_results, load_features

sns.set_theme(style="whitegrid", context="notebook")
PALETTE = {"HC": "#2a9d8f", "AP": "#e76f51", "NAP": "#e9c46a"}

In [ ]:
D = paths.DERIVED / "hierarchy"
S = pd.read_csv(D / "subject_measures.csv")
G = pd.read_csv(D / "group_tests.csv")
GN = pd.read_csv(D / "group_tests_networks.csv")
C = pd.read_csv(D / "symptom_correlations.csv")
A = pd.read_csv(D / "irreversibility_adjusted.csv")
display(G.round(4))

## Group differences

In [ ]:
show = ["Dhat", "sd_P", "GEC_asym", "FC_mean", "irr_lag1", "irr_lag2", "irr_lag3", "irr_hier_lag2"]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, m in zip(axes.ravel(), show):
    sns.boxplot(data=S, x="group", y=m, order=GROUPS, palette=PALETTE, ax=ax, showfliers=False)
    sns.stripplot(data=S, x="group", y=m, order=GROUPS, color="k", alpha=.35, size=3, ax=ax)
    r = G.set_index("measure").loc[m]
    ax.set_title(f"{m}\nF p={r.p_perm:.4f}, q={r.q_FDR:.4f}"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

## Effect sizes per Yeo network

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, kind in zip(axes, ("fdt_P", "irr_lag2")):
    d = GN[GN.measure == kind].melt(id_vars="network", value_vars=["g_AP_vs_HC", "g_NAP_vs_HC"], var_name="contrast", value_name="g")
    sns.barplot(data=d, x="network", y="g", hue="contrast", ax=ax, palette=[PALETTE["AP"], PALETTE["NAP"]])
    ax.axhline(0, color="k", lw=.8); ax.set_title(kind); ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("Hedges g vs HC")
plt.tight_layout(); plt.show()
display(GN.round(4))

## Nodal irreversibility (lag 2): NAP − HC on the brain

In [ ]:
from nilearn import plotting
feats = load_features()
atlas = load_model_results()["atlas"]
nodal = {g: np.array([feats[g][k]["irr_nodal"][2] for k in feats[g]]) for g in GROUPS}
pooled_sd = np.sqrt((nodal["HC"].var(0, ddof=1) + nodal["NAP"].var(0, ddof=1)) / 2)
d = (nodal["NAP"].mean(0) - nodal["HC"].mean(0)) / pooled_sd
plotting.plot_markers(d, atlas["coords"], node_cmap="RdBu_r", node_vmin=-1.2, node_vmax=1.2,
                      title="Cohen d, nodal irreversibility (lag 2), NAP − HC", display_mode="lzry")
plt.show()

## Is it just weaker FC? Irreversibility vs mean FC and vs model asymmetry

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(data=S, x="FC_mean", y="irr_lag2", hue="group", palette=PALETTE, ax=axes[0])
sns.scatterplot(data=S, x="GEC_asym", y="irr_lag2", hue="group", palette=PALETTE, ax=axes[1])
for ax in axes: ax.set_yscale("log")
plt.tight_layout(); plt.show()
print("Group effect after adjusting log-irreversibility for mean FC and lag-1 autocorrelation:")
display(A.round(4))

## Symptom correlations (patients, partial Spearman controlling AP/NAP)

In [ ]:
display(C.sort_values("p_perm").round(4))
P = S[S.group != "HC"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, t in zip(axes, ("PANSS", "YMRS")):
    sns.scatterplot(data=P, x="irr_lag2", y=t, hue="group", palette=PALETTE, ax=ax); ax.set_xscale("log")
plt.tight_layout(); plt.show()